In [ ]:
!pip install -q datasets matplotlib pillow

In [ ]:
import torch

print("PyTorch version:", torch.__version__)

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from datasets import load_dataset

train_stream = load_dataset(
    "hassanjbara/AI4MARS",
    split="train",
    streaming = True
)

print("Dataset loaded successfully")

In [ ]:
sample = None

for row in train_stream:
  if row["has_labels"] and row["label_mask"] is not None:
    sample = row
    break

print("Available columns:", sample.keys())

print("Has labels:", sample["has_labels"])

print("Has masks:", sample["has_masks"])

In [ ]:
from IPython.display import display

display(sample["image"])

In [ ]:
import numpy as np
from PIL import Image

mask = np.array(sample["label_mask"])

if mask.ndim == 3:
  mask = mask[..., 0]

print("Mask shape: ",mask.shape)
print("Unique pixel values: ",np.unique(mask))

In [ ]:
colors = np.array([
    [164,117,74], #Soil
    [120,130,140],  #Bedrock
    [238,190,70], #Sand
    [220,60,60] #Big Rock
],dtype=np.uint8)

colored_mask = np.zeros(
    (mask.shape[0],mask.shape[1],3),
    dtype=np.uint8
)

for class_id in range(4):
  colored_mask[mask == class_id] = colors[class_id]
colored_mask[mask==255] = [0,0,0]
display(Image.fromarray(colored_mask))

In [ ]:
image = np.array(sample["image"].convert("RGB"))

valid = mask != 255

overlay = image.copy()

overlay[valid] = (
    0.55 * image[valid] +
    0.45 * colored_mask[valid]
).astype(np.uint8)

display(Image.fromarray(overlay))

In [ ]:
values,counts = np.unique(
    mask,
    return_counts=True
)
class_names = {
    0: "Soil",
    1: "Bedrock",
    2: "Sand",
    3: "Big Rock",
    255: "Unlabelled"
}

total_pixels = mask.size

print("Total pixels:", total_pixels)
print("\nPixel distribution:\n")

for value, count in zip(values,counts):
  percentage = (count/total_pixels)*100

  print(
      f"{class_names[value]:12s} | "
      f"Pixels: {count:8d} | "
      f"Percentage: {percentage:.2f}%"
  )


In [ ]:
import numpy as np

for key in ("rover_mask","range_mask"):

  current_mask = sample[key]

  print("\nMask name:",key)

  if current_mask is None:
    print("Mask is missing")
    continue

  mask_array = np.array(current_mask)

  print("Original shape:",mask_array.shape)
  print("Data type:",mask_array.dtype)

  if mask_array.ndim == 3:
        mask_array = mask_array[..., 0]

  print("Processed shape:", mask_array.shape)

  values, counts = np.unique(
        mask_array,
        return_counts=True
    )

  print("Unique values:", values)

  print("Pixel counts:", counts)

  print(
        "Percentage of pixels to ignore:",
        round(np.mean(mask_array > 0) * 100, 2),
        "%"
    )

In [ ]:
from PIL import Image
import numpy as np

rover_mask = np.array(sample["rover_mask"])

range_mask = np.array(sample["range_mask"])

print("ROVER MASK")

display(
    Image.fromarray(
        (rover_mask * 255).astype(np.uint8)
    )
)

print("RANGE MASK")

display(
    Image.fromarray(
        (range_mask * 255).astype(np.uint8)
    )
)

In [ ]:
label = np.array(sample["label_mask"])

if label.ndim == 3:
    label = label[..., 0]

rover_ignore = rover_mask > 0
range_ignore = range_mask > 0

valid_before = label != 255

rover_affected = np.sum(
    valid_before & rover_ignore
)

range_affected = np.sum(
    valid_before & range_ignore
)

print("Labelled pixels before preprocessing:",
      np.sum(valid_before))

print("Labelled pixels inside rover region:",
      rover_affected)

print("Labelled pixels inside range region:",
      range_affected)

In [ ]:
import numpy as np

IGNORE = 255

def prepare(row):

    image = np.array(
        row["image"].convert("RGB")
    )

    label = np.array(row["label_mask"])

    if label.ndim == 3:
        label = label[..., 0]

    label = label.astype(np.int64, copy=True)

    if image.shape[:2] != label.shape:
        raise ValueError("Image and label dimensions do not match")

    for key in ("rover_mask", "range_mask"):

        if row[key] is not None:

            mask = np.array(row[key])

            if mask.ndim == 3:
                mask = mask[..., 0]

            if mask.shape != label.shape:
                raise ValueError(f"{key} dimensions do not match label")

            label[mask > 0] = IGNORE

    return image, label

In [ ]:
processed_image, processed_label = prepare(sample)

print("IMAGE INFORMATION")

print("Shape:", processed_image.shape)
print("Data type:", processed_image.dtype)

print("\nLABEL INFORMATION")

print("Shape:", processed_label.shape)
print("Data type:", processed_label.dtype)

print("Unique values:", np.unique(processed_label))

print("\nPIXEL COUNTS")

values, counts = np.unique(
    processed_label,
    return_counts=True
)

for value, count in zip(values, counts):

    print(f"Class {value}: {count} pixels")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(processed_image)
plt.title("Processed Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(processed_label)
plt.title("Processed Label Mask")
plt.axis("off")

plt.subplot(1, 3, 3)

plt.imshow(processed_image)
plt.imshow(
    np.ma.masked_where(
        processed_label == IGNORE,
        processed_label
    ),
    alpha=0.45,
    cmap="jet",
    vmin=0,
    vmax=3
)

plt.title("Processed Overlay")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image
import numpy as np

toy_row = {

    "image": Image.fromarray(
        np.zeros((2, 3, 3), dtype=np.uint8)
    ),

    "label_mask": np.array([
        [0, 1, 2],
        [3, 255, 0]
    ], dtype=np.uint8),

    "rover_mask": np.array([
        [0, 1, 0],
        [0, 0, 0]
    ], dtype=np.uint8),

    "range_mask": np.array([
        [0, 0, 0],
        [1, 0, 0]
    ], dtype=np.uint8)
}

In [ ]:
toy_image, toy_label = prepare(toy_row)

print("Original label:")
print(toy_row["label_mask"])

print("\nProcessed label:")
print(toy_label)

In [ ]:
expected = np.array([
    [0, 255, 2],
    [255, 255, 0]
], dtype=np.int64)

assert np.array_equal(toy_label, expected)

assert toy_image.shape == (2, 3, 3)

assert toy_label.dtype == np.int64

print("All preprocessing tests passed!")

In [ ]:
!df -h /content

# Training and Validation Preparation

In [ ]:
from datasets import load_dataset

train_data = load_dataset(
    "hassanjbara/AI4MARS",
    split="train"
)

print("Dataset loaded successfully!")

print("Dataset type:", type(train_data))

print("Total training images:", len(train_data))

print("Available columns:", train_data.column_names)

In [ ]:
has_labels = train_data["has_labels"]

labelled_indices = [
    index
    for index, has_label in enumerate(has_labels)
    if has_label
]

unlabelled_count = len(train_data) - len(labelled_indices)

print("Total images:", len(train_data))

print("Images containing labels:", len(labelled_indices))

print("Images without labels:", unlabelled_count)

In [ ]:
labelled_data = train_data.select(labelled_indices)

print("Labelled dataset created!")

print("Total samples:", len(labelled_data))

print("Dataset type:", type(labelled_data))

In [ ]:
splits = labelled_data.train_test_split(
    test_size=0.10,
    shuffle=True,
    seed=42
)

train_dataset = splits["train"]

val_dataset = splits["test"]

In [ ]:
print("TRAINING DATASET")
print("Images:", len(train_dataset))

print("\nVALIDATION DATASET")
print("Images:", len(val_dataset))

print("\nTOTAL")
print("Images:", len(train_dataset) + len(val_dataset))

In [ ]:
print(
    "All training samples labelled:",
    all(train_dataset["has_labels"])
)

print(
    "All validation samples labelled:",
    all(val_dataset["has_labels"])
)

In [ ]:
import numpy as np
from PIL import Image

sample_index = 10

row = train_dataset[sample_index]

image, label = prepare(row)

print("Image shape:", image.shape)

print("Label shape:", label.shape)

print("Unique label values:", np.unique(label))

display(Image.fromarray(image))